Then, we calculated alpha diversity, as measured by Richness, Evenness, Simpson index, and Shannon entropy in the R package vegan (v.2.6.8). Beta diversity, as measured by Bray-Curtis distance in the R package vegan (v.2.6.8).

In [ ]:
rm(list=ls())
library(tidyverse)
library(ggplot2)
library(ggsci)
library(dplyr)
library(patchwork)
library(Maaslin2)
library(vegan)
library(edgeR)

count_file_name="E:/QZ/01_metag/00_Rawdata/01_metaphlan/merged_abundance_table_read_count_species.txt"
rawdata = read.table(file             = count_file_name,
                     header           = TRUE,
                     sep              = "\t", 
                     row.names        = 1,
                     stringsAsFactors = FALSE,
                     check.names = F)

Prevalence_0.05_species_list<-read.csv("E:/QZ/01_metag/01_Prevalence_0.05/Prevalence_0.05_species_list.csv")

data <-data.frame(edgeR::cpm(rawdata),check.names = FALSE)

data$Phylum<-rownames(data)
df<-data%>%gather(sampleID,cpm,-Phylum)%>%arrange(cpm)
head(df)
# Compute pathway alpha divertities per sample
result<-df %>%
  group_by(sampleID) %>%
  summarise(shannons_div = vegan::diversity(cpm),
            J = diversity(cpm)/log(specnumber(cpm)),
            simpson_div = vegan::diversity(cpm,index = "simpson"),
            num_pathways = sum(cpm>0)
  )

In [ ]:
rm(list=ls())
library(tidyverse)
library(ggplot2)
library(ggsci)
library(dplyr)
library(patchwork)
library(microbiome)
library(vegan)
library(ecodist)

count_file_name="E:/QZ/01_metag/00_Rawdata/01_metaphlan/merged_abundance_table_read_count_species.txt"

feature <- read.table(count_file_name,sep = "\t",header = 1,
                      row.names=1,check.names=F)

meta_data<-read.csv("E:/QZ/UniqueID_final_v2.csv",row.names = 1)%>%
  filter(Treat !="Used")%>%
  filter(Group =="Control")%>%
  filter(Metagenome %in% colnames(feature))%>%
  mutate(Age_group=ifelse(Age>79,"D80",
                          ifelse(Age>69,"D70",
                                 ifelse(Age>59,"D60",
                                        ifelse(Age>49,"D50",
                                               ifelse(Age>39,"D40",
                                                      ifelse(Age>29,"D30","D20")))))))
table(meta_data$Age_group)

species_subset <- read.csv("E:/QZ/01_metag/01_Prevalence_0.05/Prevalence_0.05_species_list.csv")
length(species_subset$body_site)
bray_curtis_dist <- vegan::vegdist(t(feature[species_subset$x,meta_data$Metagenome]), method = "bray")
# bray_curtis_dist <- vegan::vegdist(t(feature[,meta_data$Metagenome]), method = "bray")

bray_curtis_pcoa <- ecodist::pco(bray_curtis_dist)
bray_curtis_pcoa